<a href="https://colab.research.google.com/github/consor92/Probabilidad_aplicada/blob/Clase_Init_R/Informa_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 4. Variables Aleatorias Discretas: Ensayos de Bernoulli y Distribución Binomial

### Ensayos de Bernoulli

Un **ensayo de Bernoulli** es un experimento aleatorio que tiene solo dos resultados posibles: "éxito" o "fracaso". Estos resultados son mutuamente excluyentes y exhaustivos.

**Características clave de un ensayo de Bernoulli:**

1.  **Dos resultados posibles:** Siempre hay un resultado designado como "éxito" (generalmente representado por $E$) y el otro como "fracaso" (representado por $F$).
2.  **Probabilidad constante de éxito ($p$):** La probabilidad de que ocurra un "éxito" es fija y se denota como $p$. Consecuentemente, la probabilidad de "fracaso" es $1-p$ (a menudo denotada como $q$).
3.  **Independencia:** Si se realizan múltiples ensayos de Bernoulli, el resultado de un ensayo no afecta el resultado de los otros ensayos.

### Distribución Binomial

La **distribución binomial** surge cuando se realizan una secuencia de $n$ ensayos de Bernoulli independientes e idénticos. Esta distribución modela el número de éxitos en $n$ ensayos.

Para que una situación pueda modelarse como una distribución binomial, debe cumplir con los siguientes requisitos:

1.  **Cantidad fija de ensayos ($n$):** El número de veces que se repite el experimento (ensayo) es fijo y predeterminado.
2.  **Dos resultados posibles:** Cada ensayo individual solo puede resultar en "éxito" o "fracaso".
3.  **Probabilidad de éxito constante ($p$):** La probabilidad de éxito ($p$) es la misma para cada ensayo.
4.  **Ensayos independientes:** El resultado de un ensayo no influye en el resultado de ningún otro ensayo.

### Ejemplo en el Marco de un Proyecto

Consideremos un escenario para aplicar estos conceptos. Si tu proyecto involucra, por ejemplo, el rendimiento de un proceso productivo, podríamos definir:

*   **Ensayo:** La inspección de una unidad de producto.
*   **Éxito:** La unidad de producto es **defectuosa**.
*   **Fracaso:** La unidad de producto es **no defectuosa**.
*   **Probabilidad de éxito ($p$):** La probabilidad de que una unidad de producto sea defectuosa, basada en datos históricos o especificaciones del proceso.
*   **Número de ensayos ($n$):** El número total de unidades de producto inspeccionadas en un lote o turno.

## 4. Modelación Probabilística: Distribución Binomial en Tráfico de Red

### 1. Justificación Técnica
En el análisis de redes, los flujos se dividen en dos categorías fundamentales:
*   **Tráfico de Control (Overhead):** Paquetes sin carga útil (`Fwd Act Data Pkts = 0`), usados para abrir conexiones (Handshake), confirmar recepciones (ACK) o mantener enlaces.
*   **Tráfico de Carga Útil (Payload):** Flujos con datos reales (`Fwd Act Data Pkts > 0`) que transportan la información del usuario.

**¿Por qué modelarlo?** Establecer un comportamiento base permite detectar anomalías. Si la probabilidad de encontrar flujos con datos cae drásticamente, el modelo binomial alertará sobre posibles ataques de denegación de servicio (DoS) o escaneos de puertos.

---

### 2. Validación de Requisitos del Modelo
Para aplicar la **Distribución Binomial**, el caso de estudio cumple con los cuatro requisitos fundamentales:

1.  **Cantidad fija de ensayos ($n$):** Se analiza un lote cerrado y predeterminado de flujos (ej. $n = 100$).
2.  **Dos resultados posibles (Binarización):**
    *   **Éxito ($1$):** El flujo lleva carga útil ($Fwd\ Act\ Data\ Pkts > 0$).
    *   **Fracaso ($0$):** El flujo es solo de control ($Fwd\ Act\ Data\ Pkts = 0$).
3.  **Probabilidad de éxito constante ($p$):** Se calcula como la proporción histórica de flujos con datos en el dataset.
4.  **Ensayos independientes:** Se asume que el estado de un flujo no influye en el comportamiento del siguiente.

---

### 3. Definición del Modelo Matemático

**Variable Aleatoria:**
$X$: Número de flujos que transportan carga útil en una muestra de tamaño $n$.

**Distribución Teórica:**
$$X \sim B(n, p)$$

**Función de Masa de Probabilidad (FMP):**
La probabilidad de observar exactamente $x$ éxitos es:
$$P(X = x) = \binom{n}{x} p^x (1-p)^{n-x}$$

Donde el coeficiente combinatorio se define como:
$$\binom{n}{x} = \frac{n!}{x!(n-x)!}$$

**Indicadores Estadísticos:**
*   **Valor Esperado (Media):** $E(X) = n \cdot p$
*   **Varianza:** $V(X) = n \cdot p \cdot (1-p)$

---

### 4. Análisis del Mecanismo Técnico (Contexto de Red)
Es crucial entender el origen de estos valores según los protocolos TCP/UDP:

#### ¿Por qué ocurre un Fracaso ($x=0$)?
*   **Handshake y Control:** Paquetes SYN/ACK de apertura o cierre de conexión sin envío de datos.
*   **Escaneo de Puertos:** Intentos de conexión maliciosos que solo buscan puertos abiertos.
*   **Mensajes ICMP:** Mensajes de diagnóstico (como el *Ping*) que no llevan carga de aplicación.

#### ¿Por qué ocurre un Éxito ($x>0$)?
*   **Solicitudes Atómicas ($1$ paquete):** Consultas DNS o peticiones HTTP simples.
*   **Transferencia de Datos ($>1$ paquete):** Cuando la información supera el Tamaño Máximo de Segmento (MSS, ~1460 bytes), el protocolo fragmenta la carga en múltiples paquetes.

---

### 5. Próximos Pasos Prácticos
1.  **Calcular $p$:** Dividir el total de filas con datos entre el total del dataset.
2.  **Definir escenarios:** Establecer un $n$ (ej. $50$) y calcular probabilidades acumuladas para reportar hallazgos.

In [5]:
# Ahora que Python abrió la carpeta, R sí podrá listar los archivos
if(!require(data.table)) install.packages("data.table")
library(data.table)

# Prueba la Ruta 1 (Tradicional)
ruta_1 <- "/content/drive/MyDrive/tu_archivo_red.csv"

# Intenta leer directamente. Si no funciona, intentará con la Ruta 2 de forma automática.
datos_red <- tryCatch({
    cat("Intentando cargar desde MyDrive...\n")
    fread(ruta_1)
}, error = function(e) {
    cat("Ruta 1 no disponible. Intentando desde 'My Drive' (con espacio)...\n")
    ruta_2 <- "/content/drive/My Drive/tu_archivo_red.csv"
    return(fread(ruta_2))
})

cat("\n¡Logrado! El dataset se cargó correctamente.\n")
cat("Total de filas leídas:", nrow(datos_red), "\n")
print(head(datos_red, 3)) # Muestra las primeras 3 filas para verificar

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




Intentando cargar desde MyDrive...
Ruta 1 no disponible. Intentando desde 'My Drive' (con espacio)...


Taking input= as a system command because it contains a space ('/content/drive/My Drive/tu_archivo_red.csv'). If it's a filename please remove the space, or use file= explicitly. A variable is being passed to input= and when this is taken as a system command there is a security concern if you are creating an app, the app could have a malicious user, and the app is not running in a secure environment; e.g. the app is running as root. Please read item 5 in the NEWS file for v1.11.6 for more information and for the option to suppress this message.

Warning message in (if (.Platform$OS.type == "unix") system else shell)(paste0("(", :
“error in running command”


ERROR: Error in fread(ruta_2): External command failed with exit code 127. This can happen when the disk is full in the temporary directory ('/tmp/RtmpY6qr2k'). See ?fread for the tmpdir argument.
